# Final runs on Kaggle: M3 and M4b again, now saving probabilities

Phase 5 needs each model's **class probabilities** on the dev set and on the
**unlabelled** test file, so we can average seeds and combine models (chosen on dev),
and score the test set once at the very end. The earlier M3/M4b runs saved only dev
scores, so this notebook trains them again with `--test`:

| Run | Command | Time on a T4 |
|---|---|---|
| M3: MuRIL + two heads, multi-task | `--task multi` (5 seeds × 5 epochs) | about 25 minutes |
| M4b: MuRIL + pair head | `--task pairs --pair_weights product --epochs 8` | about 36 minutes |

All settings are the same as before. At each seed's best dev epoch the script also saves
probabilities in `results/probs/<run name>/{dev,test}_seed<N>.csv`. The test file has
**no labels**, so nothing is scored here. GPU results are not bit-for-bit repeatable, so
the dev scores will be close to, but not exactly, the earlier ones.

## Setup on Kaggle
1. **Update your private dataset.** Dataset `training-1` → *New Version* → upload
   `tulu-hope-kaggle-dataset-final.zip` → *Create*. It contains train, dev, the
   **unlabelled** test file (`multitask_test_withoutlabel.csv`, Id + text only),
   `train_multitask.py`, `train_char.py` and `probs_io.py`. The labelled test file is never uploaded.
2. **Import this notebook.** *File* → *Import Notebook* → `notebooks/final_probs_kaggle.ipynb`.
   Make sure the notebook uses the newest dataset version (*Input* panel).
3. **Session options:** *Accelerator* → **GPU T4 x2**; *Internet* → **On**.
4. **Quick test first.** Set `QUICK_TEST = True` → *Restart & Run All* (about 3–4 minutes).
   Check that Step 1 says `(new: has --probs_dir)` and that both runs print
   `test (unlabelled)=744: probabilities will be saved`.
   Then set it back to `False` → *Save Version* → *Save & Run All (Commit)* (about 1 hour).

## Bringing the results back to the Mac
When the version is *Complete*: *Output* tab → download `final_probs_outputs.zip` →
put it in the project's `results/` folder and tell Claude.

In [ ]:
# Step 1: find the input files and choose where outputs go.
from pathlib import Path
import torch

ON_KAGGLE = Path("/kaggle/input").exists()

# Set to True for a quick check (1 seed, 1 epoch per run). Set to False for the real run.
QUICK_TEST = False

if ON_KAGGLE:
    def find(name):
        # rglob searches every sub-folder of /kaggle/input for a file with this name.
        matches = sorted(Path("/kaggle/input").rglob(name))
        if not matches:
            raise FileNotFoundError(f"{name} not found. Did you attach the NEW version of the dataset?")
        return matches[0]

    # The input folder holds copies of the script from older dataset versions.
    # Use the copy that knows --probs_dir AND has probs_io.py next to it.
    scripts = sorted(Path("/kaggle/input").rglob("train_multitask.py"))
    good = []
    for s in scripts:
        ok = "--probs_dir" in s.read_text() and (s.parent / "probs_io.py").exists()
        print("found script:", s, "(new: has --probs_dir)" if ok else "(OLD)")
        if ok:
            good.append(s)
    if not good:
        raise FileNotFoundError(
            "No train_multitask.py with --probs_dir (and probs_io.py). Upload "
            "tulu-hope-kaggle-dataset-final.zip as a NEW VERSION of the dataset, then make "
            "sure this notebook uses that version (right-hand panel -> Input).")
    SCRIPT = good[0]
    TRAIN = find("train_clean.csv")
    DEV = find("dev_multitask.csv")
    TEST = find("multitask_test_withoutlabel.csv")   # UNLABELLED: Id + text only
    OUT = Path("/kaggle/working/final_outputs")      # /kaggle/working is the folder Kaggle keeps
else:
    SCRIPT = Path("../scripts/train_multitask.py")
    TRAIN = Path("../data/train_clean.csv")
    DEV = Path("../data/dev_multitask.csv")
    TEST = Path("../data/multitask_test_withoutlabel.csv")
    OUT = Path("..")

RESULTS = OUT / "results"
CKPT = OUT / "checkpoints"
PROBS = RESULTS / "probs"
LOGS = RESULTS / "logs"
LOGS.mkdir(parents=True, exist_ok=True)

# In a quick test these options come LAST on the command line, so they override
# the --epochs given before them (argparse keeps the last value it sees).
QUICK = "--seeds 1 --epochs 1" if QUICK_TEST else ""
M3_NAME = "m3_muril-base-cased_multi" + ("_ep1" if QUICK_TEST else "")
M4B_NAME = "m4_muril-base-cased_pairs_w-product_ep" + ("1" if QUICK_TEST else "8")

print("script :", SCRIPT)
print("train  :", TRAIN)
print("dev    :", DEV)
print("test   :", TEST, "(unlabelled)")
print("outputs:", OUT)
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU only)")

In [ ]:
# Step 1b: check that MuRIL can be downloaded (needs Internet = On).
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("google/muril-base-cased")
print("MuRIL tokenizer OK. Example:", tok.tokenize("baari porlu movie"))

## Step 2: the two runs
A line starting with `!` is a shell command; `{SCRIPT}` etc. insert the Python variables from Step 1.

In [ ]:
# M3: MuRIL + two heads, multi-task (same settings as Phase 3), now with probabilities
!python -u {SCRIPT} --task multi --train {TRAIN} --dev {DEV} --test {TEST} --out_dir {RESULTS} --ckpt_dir {CKPT} --probs_dir {PROBS} {QUICK} 2>&1 | tee {LOGS}/{M3_NAME}.log

In [ ]:
# M4b: MuRIL + pair head, product pair weights, 8 epochs (same settings as Phase 4), now with probabilities
!python -u {SCRIPT} --task pairs --pair_weights product --epochs 8 --train {TRAIN} --dev {DEV} --test {TEST} --out_dir {RESULTS} --ckpt_dir {CKPT} --probs_dir {PROBS} {QUICK} 2>&1 | tee {LOGS}/{M4B_NAME}.log

In [ ]:
# Step 3: dev scores of the two runs, and a check that all probability files exist.
import json
import pandas as pd

earlier = {M3_NAME: "earlier: tone 0.540, type 0.426", M4B_NAME: "earlier: tone 0.523, type 0.437"}
rows = []
for name in (M3_NAME, M4B_NAME):
    r = json.loads((RESULTS / f"{name}.json").read_text())
    files = sorted((PROBS / name).glob("*.csv"))
    rows.append({"run": name, "tone F1": r["tone_mean"], "tone std": r["tone_std"],
                 "type F1": r["type_mean"], "type std": r["type_std"],
                 "best epochs": r["best_epochs"], "probability files": len(files),
                 "compare": earlier[name]})
    for i, seed_scores in enumerate(r["per_class_f1_by_seed"]):
        for head, classes in seed_scores.items():
            for cls, f1 in classes.items():
                if f1 == 0.0:
                    print(f"WARNING {name}: seed {r['seeds'][i]} has 0.000 F1 for {head} class '{cls}'")

print("Expected probability files per run: 2 per seed (dev + test) -> 10 for 5 seeds, 2 in a quick test")
pd.DataFrame(rows).round(3)

In [ ]:
# Step 4 (Kaggle only): pack results, probabilities and checkpoints into one zip for the Output tab.
import shutil

if ON_KAGGLE:
    zip_path = shutil.make_archive("/kaggle/working/final_probs_outputs", "zip", root_dir=OUT)
    print("created", zip_path, "- download it from the Output tab")
else:
    print("Running locally: results are already in the project's results/ folder.")